In [5]:

# Step 1: Imports
from getpass import getpass
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document

# Step 2: Load Gemini API key
GOOGLE_API_KEY = getpass("Enter Gemini API Key: ")

# Step 3: Setup LLM and Embeddings
llm = ChatGoogleGenerativeAI(
    temperature=0,
    model="gemini-3.5-flash",
    google_api_key=GOOGLE_API_KEY
)

embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

# Step 4: Load and split document
with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=300,
    chunk_overlap=50
)

texts = splitter.split_text(raw_text)
documents = [Document(page_content=t) for t in texts]

# Step 5: Vector Store
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

print("Base setup complete.")


Enter Gemini API Key: ··········
Base setup complete.


In [2]:
%pip install -q --no-deps langchain-google-genai==4.4.3

In [3]:
%pip install -q langchain-community==0.4.2 faiss-cpu
%pip install -q filetype

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-google-genai 4.4.3 requires google-genai>=2.20.0, but you have google-genai 2.12.1 which is incompatible.
langchain-google-genai 4.4.3 requires langchain-core<2.0.0,>=1.7.5, but you have langchain-core 1.7.4 which is incompatible.


In [6]:

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Use the following context to answer the question:\n\n"
    "{context}\n\n"
    "Question: {question}"
)

stuff_chain = create_stuff_documents_chain(
    llm,
    prompt,
    document_variable_name="context"
)

response = stuff_chain.invoke({
    "context": documents[:3],
    "question": "What is this document about?"
})

print("\nStuffDocumentsChain Answer:", response)



StuffDocumentsChain Answer: Based on the provided context, the document is about **LangChain**—a framework created by Harrison Chase for building applications with LLMs. It outlines what LangChain supports (such as RAG, agents, memory, and tools) and its common use cases (chatbots, document Q&A, and AI workflows).


In [7]:

from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

# Initial summary prompt
initial_prompt = PromptTemplate.from_template("""
Write a concise summary of the following text:

{context}
""")

# Refine prompt
refine_prompt = PromptTemplate.from_template("""
We have an existing summary:
"{existing_answer}"

Refine the summary with this new context:
"{context}"

If the context isn't useful, return the original summary.
""")

# Set up individual chains
initial_summary_chain = initial_prompt | llm | StrOutputParser()
refine_summary_chain = refine_prompt | llm | StrOutputParser()

# Start with first chunk
summary = initial_summary_chain.invoke({
    "context": documents[0].page_content
})

# Iteratively refine with remaining documents
for doc in documents[1:]:
    summary = refine_summary_chain.invoke({
        "existing_answer": summary,
        "context": doc.page_content
    })

# Output final summary
print("Refined Summary:\n")
print(summary)


Refined Summary:

Created by Harrison Chase, LangChain is a framework for building LLM applications—such as chatbots, document Q&A, and AI workflows—that supports RAG, agents, memory, and tools.
